# 06 — Business Impact Analysis
## What is the recommender worth in rupees?

**Goal:**
Translate model metrics into business value.<br>
This is what DS roles at MAANG actually do —<br>
connect model performance to revenue impact.<br>

**Framework:**
1. Engagement lift  → how many more relevant movies found
2. Retention value  → users who find good movies stay longer
3. ROI calculation  → cost of building vs value generated
4. Sensitivity analysis → how robust are our assumptions?

In [1]:
#importing libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import os
import json
import pickle 
import seaborn as sns
from matplotlib.ticker import mticker
from datetime import datetime

print("Libraries loaded.")
print(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

ImportError: cannot import name 'mticker' from 'matplotlib.ticker' (c:\Projects\Cinemate V2\.venv\Lib\site-packages\matplotlib\ticker.py)

In [ ]:
#Getting the project root path 
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")
MODELS_DIR = os.path.join(BASE_DIR, "models")
PLOT_DIR = os.path.join(PROCESSED_DATA_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

print(f"Base directory: {BASE_DIR}")

In [ ]:
#Load processed directories
print("Loading preprocessed file...............")


with open(os.path.join(MODELS_DIR, "ab_test_results.json"), "r") as f:
    ab_results = json.load(f)

with open(os.path.join(MODELS_DIR, "two_tower_results.json"), "r") as f:
    tt_results = json.load(f)

with open(os.path.join(PROCESSED_DATA_DIR, "dataset_constants.pkl"), "rb") as f:
    constants = pickle.load(f)

NUM_USERS = constants['NUM_USERS']
print(f"Business impact analysis for {NUM_USERS} users")

### Assumptions

In [ ]:
print("=" * 55)
print("BUSINESS ASSUMPTIONS")
print("=" * 55)
print()
print("These are conservative industry estimates.")
print("Each assumption is cleary stated and adjustable.")
print()

#Assumption 
ASSUMPTIONS = {
    # Platform metrics
    "monthly_active_users"   : 100_000,
    "avg_revenue_per_user"   : 199,      

    # Engagement metrics
    "baseline_retention_rate": 0.72,     # 72% monthly retention
    "rec_retention_lift"     : 0.05,     # +5% from recommendations
    "baseline_ctr"           : 0.08,     # 8% click-through on suggestions
    "rec_ctr_lift"            : 0.15,    # +15% CTR improvement

    # Model metrics (from our results)
    "ndcg_random"            : ab_results['control_ndcg']['mean'],
    "ndcg_two_tower"         : ab_results['treatment_ndcg']['mean'],
    "relative_lift"          : ab_results['lift']['relative_pct'],

    # Development cost
    "dev_cost_inr"           : 500_000,  # ₹5L to build
    "infra_cost_monthly"     : 10_000,   # ₹10K/month compute
}

for k, v in ASSUMPTIONS.items():
    if isinstance(v, float):
        print(f"  {k:<35s}: {v:.4f}")
    else:
        print(f"  {k:<35s}: {v}")

### Revenue impact


In [ ]:
print("=" * 58)
print("REVENUE IMPACT CALCULATION")
print("=" * 58)
print()

MAU          = ASSUMPTIONS['monthly_active_users']
ARPU         = ASSUMPTIONS['avg_revenue_per_user']
base_ret     = ASSUMPTIONS['baseline_retention_rate']
ret_lift     = ASSUMPTIONS['rec_retention_lift']
rel_lift_pct = ASSUMPTIONS['relative_lift']

# Monthly revenue baseline
monthly_revenue_base = MAU * ARPU
print(f"Monthly revenue (baseline) : ₹{monthly_revenue_base:,.0f}")

# Users retained additionally due to recommendations
additional_retained = MAU * ret_lift
monthly_rec_revenue = additional_retained * ARPU
print(f"Additional users retained : {additional_retained:,.0f}")
print(f"Monthly rec revenue lift : ₹{monthly_rec_revenue:,.0f}")

# Annual impact
annual_lift = monthly_rec_revenue * 12
dev_cost    = ASSUMPTIONS['dev_cost_inr']
infra_annual= ASSUMPTIONS['infra_cost_monthly'] * 12
net_annual  = annual_lift - dev_cost - infra_annual
roi         = (net_annual / (dev_cost + infra_annual)) * 100

print()
print(f"Annual revenue lift : ₹{annual_lift:,.0f}")
print(f"Development cost : ₹{dev_cost:,.0f}")
print(f"Annual infrastructure cost : ₹{infra_annual:,.0f}")
print(f"Net annual value : ₹{net_annual:,.0f}")
print(f"ROI : {roi:.1f}%")
print()

# Payback period
payback_months = (dev_cost + infra_annual) / monthly_rec_revenue
print(f"Payback period : {payback_months:.1f} months")

In [ ]:
print("Sensitivity Analysis — Revenue lift vs assumptions")
print()

retention_lifts = [0.02, 0.03, 0.05, 0.07, 0.10]
arpu_values     = [99, 149, 199, 299, 499]

# Build sensitivity matrix
matrix = np.zeros((len(retention_lifts), len(arpu_values)))
for i, ret in enumerate(retention_lifts):
    for j, arpu in enumerate(arpu_values):
        monthly = MAU * ret * arpu
        matrix[i, j] = monthly * 12 / 1_00_000  # in lakhs

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(matrix, cmap='YlGn', aspect='auto')

ax.set_xticks(range(len(arpu_values)))
ax.set_yticks(range(len(retention_lifts)))
ax.set_xticklabels([f'₹{v}' for v in arpu_values])
ax.set_yticklabels([f'{v*100:.0f}%' for v in retention_lifts])
ax.set_xlabel('Average Revenue Per User (₹/month)')
ax.set_ylabel('Retention Lift from Recommendations')
ax.set_title('Annual Revenue Lift (₹ Lakhs)\nSensitivity Analysis')

for i in range(len(retention_lifts)):
    for j in range(len(arpu_values)):
        ax.text(j, i, f'₹{matrix[i,j]:.0f}L',  ha='center', va='center', fontsize=9, fontweight='bold',
                color='black' if matrix[i,j] < matrix.max()*0.7 else 'white')

plt.colorbar(im, label='Annual Lift (₹ Lakhs)')
plt.tight_layout()
plt.savefig( os.path.join(PLOT_DIR,'11_sensitivity_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Green cells = high value scenarios")
print("Our base case is highlighted (5% retention, ₹199 ARPU)")

In [ ]:
business_results = {
    "assumptions"     : ASSUMPTIONS,
    "monthly_lift_inr": float(monthly_rec_revenue),
    "annual_lift_inr" : float(annual_lift),
    "net_annual_inr"  : float(net_annual),
    "roi_pct"         : float(roi),
    "payback_months"  : float(payback_months),
    "timestamp"       : datetime.now().strftime(
        '%Y-%m-%d %H:%M:%S'
    )
}

biz_path = os.path.join(MODELS_DIR, "business_impact.json")
with open(biz_path, "w") as f:
    json.dump(business_results, f, indent=2)

print(f"Business impact saved to {biz_path}")
print()
print("Resume line:")
print(f"  'Recommender system estimated to generate "
      f"₹{annual_lift/100000:.1f}L annual revenue lift "
      f"with {roi:.0f}% ROI based on 5% retention "
      f"improvement across 100K MAU'")